# 01. Research Framing and Data Provenance

**Project:** Early-warning analytics on the Open University Learning Analytics Dataset (OULAD)
**Course:** Advanced Data Science, Master of AI and Data Science (CADT)
**SDG link:** SDG 4, Quality Education (inclusive and equitable education, reducing dropout and failure)

This notebook covers the research framing and the provenance and structure of the data. It does no modelling.
The master table is built in `02_master_table.ipynb`.

## 1. Research framing

**Working title.** *Who is at risk after four weeks? Early engagement patterns, equity and cohort stability in a distance-learning programme.*

**Domain.** Educational data mining / learning analytics (SDG 4).

**Problem statement.** Distance-learning courses lose a large share of students to withdrawal and failure.
Support teams can only act if they know who is struggling early, while there is still time to help.
Two risks make this hard. Predictions made late in the course are easy but useless, because the outcome is almost decided.
Models trained on one cohort may also fail on the next, or may work worse for some demographic groups.
This project studies what can honestly be known about a student's risk after the first four weeks of a course, using only information available at that moment.

**Objective.** Describe and model early-course risk (Fail or Withdrawn) from the first 28 days of a course-presentation,
and test whether the findings are stable across cohorts and across student groups.

**Research questions**

1. **RQ1 (association).** How do early VLE engagement, early assessment behaviour and student background relate to ending the course at risk?
2. **RQ2 (structure).** Do distinct engagement profiles exist among students at day 28, and do their risk levels differ?
3. **RQ3 (prediction and stability).** How well can day-28 features predict risk, and does a model trained on 2013 cohorts hold up on 2014 cohorts (data drift)?
4. **RQ4 (equity).** Are risk and prediction errors unevenly distributed across deprivation band (IMD), prior education, age and disability?

**Unit of analysis.** One student in one course-presentation (`code_module` + `code_presentation` + `id_student`).
The same person can appear in several rows. This matters for splitting and for independence assumptions.

**Target population.** Students registered on OU modules who are still enrolled at day 28 of the presentation.
Students who left before the cutoff are outside the prediction problem (see notebook 02).

**Conceptual map**

| Role | Variables |
|---|---|
| Focal outcome | `final_result` collapsed to `target_at_risk` (Fail or Withdrawn = 1, Pass or Distinction = 0) |
| Explanatory (early behaviour) | VLE clicks by week, active days, activity-type mix, early assessment submission and scores |
| Explanatory (background) | gender, region, highest education, IMD band, age band, disability, previous attempts, studied credits |
| Context / confounders | module, presentation (year and Feb/Oct start), course length |
| Identifiers | `id_student`, `id_site`, `id_assessment` (never used as features) |
| Leakage risks | `final_result`, `date_unregistration`, anything recorded after day 28 |

**Expected value.** A transparent, leakage-free early-warning baseline and a clear picture of where it fails (new cohorts, specific groups).

**Main ethical consideration.** The data are anonymised, but they include sensitive attributes (disability, deprivation, age, gender, region).
A risk model must be used to offer support, never to deny opportunities. Group-level error rates must be reported (RQ4).

## 2. Provenance

| Item | Detail |
|---|---|
| Creator | The Open University (UK), Knowledge Media Institute |
| Original publication | Kuzilek, J., Hlosta, M., Zdrahal, Z. (2017). *Open University Learning Analytics dataset.* Scientific Data 4, 170171. doi:10.1038/sdata.2017.171 |
| Obtained from | UCI Machine Learning Repository, dataset id 349 (`archive.ics.uci.edu/dataset/349`). The OU site now redirects to a new portal, so the UCI mirror was used. |
| Access date | 2026-10-24 |
| Licence | CC BY 4.0 as stated by the dataset authors. Re-confirm on the source page before final submission. |
| Collection | Administrative records from the OU virtual learning environment (VLE) and student records system. Not a survey and not a sample designed for research. |
| Coverage | 7 modules, 22 module-presentations that started in 2013 and 2014 (February "B" and October "J"). UK distance-learning students. |
| Exclusions | Other modules and years. Students without a record in a selected presentation. Anything not logged by the VLE (offline study, other platforms). |

**Validity risks to carry forward**

- *Selection bias:* only 7 modules and 4 presentations, so results may not generalise to other courses.
- *Measurement:* clicks measure platform activity, not learning. Students who study offline look inactive.
- *Repeated students:* about 3,500 people appear in more than one presentation, so rows are not fully independent.
- *Leakage:* withdrawal date and the final result are post-outcome information.
- *Sensitive attributes:* deprivation, disability and age can act as proxies and need fairness checks.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

RAW = Path("../data/raw")
NA = ["?"]  # OULAD stores missing values as the string "?"

courses = pd.read_csv(RAW / "courses.csv", na_values=NA)
assessments = pd.read_csv(RAW / "assessments.csv", na_values=NA)
vle = pd.read_csv(RAW / "vle.csv", na_values=NA)
student_info = pd.read_csv(RAW / "studentInfo.csv", na_values=NA)
student_reg = pd.read_csv(RAW / "studentRegistration.csv", na_values=NA)
student_assessment = pd.read_csv(RAW / "studentAssessment.csv", na_values=NA)

# The click log is large (about 10.6M rows); only count its lines here, it is loaded in notebook 02.
with open(RAW / "studentVle.csv", "rb") as f:
    n_click_rows = sum(1 for _ in f) - 1

## 3. Structural understanding: what does one row mean?

In [2]:
inventory = pd.DataFrame([
    ("courses", len(courses), courses.shape[1], "code_module + code_presentation", "one module-presentation"),
    ("assessments", len(assessments), assessments.shape[1], "id_assessment", "one assessment in a module-presentation"),
    ("vle", len(vle), vle.shape[1], "id_site", "one VLE page/material"),
    ("studentInfo", len(student_info), student_info.shape[1], "module + presentation + id_student", "one student in one module-presentation"),
    ("studentRegistration", len(student_reg), student_reg.shape[1], "module + presentation + id_student", "registration record for the same student-presentation"),
    ("studentAssessment", len(student_assessment), student_assessment.shape[1], "id_assessment + id_student", "one submitted assessment by one student"),
    ("studentVle", n_click_rows, 6, "(none unique, see notebook 02)", "clicks by one student on one material on one day"),
], columns=["table", "rows", "cols", "key", "one row is"])
inventory

,table,rows,cols,key,one row is
0,courses,22,3,code_module + code_presentation,one module-presentation
1,assessments,206,6,id_assessment,one assessment in a module-presentation
2,vle,6364,6,id_site,one VLE page/material
3,studentInfo,32593,12,module + presentation + id_student,one student in one module-presentation
4,studentRegistration,32593,5,module + presentation + id_student,registration record for the same student-prese...
5,studentAssessment,173912,5,id_assessment + id_student,one submitted assessment by one student
6,studentVle,10655280,6,"(none unique, see notebook 02)",clicks by one student on one material on one day


In [3]:
KEY = ["code_module", "code_presentation", "id_student"]

checks = {
    "studentInfo key unique": not student_info.duplicated(KEY).any(),
    "studentRegistration key unique": not student_reg.duplicated(KEY).any(),
    "studentInfo and studentRegistration match 1:1": len(student_info.merge(student_reg, on=KEY, how="outer", indicator=True).query("_merge != 'both'")) == 0,
    "studentAssessment (assessment, student) unique": not student_assessment.duplicated(["id_assessment", "id_student"]).any(),
    "assessments id unique": assessments.id_assessment.is_unique,
    "vle id_site unique": vle.id_site.is_unique,
}
pd.Series(checks, name="passed").to_frame()

,passed
studentInfo key unique,True
studentRegistration key unique,True
studentInfo and studentRegistration match 1:1,True
"studentAssessment (assessment, student) unique",True
assessments id unique,True
vle id_site unique,True


In [4]:
n_rows = len(student_info)
n_people = student_info.id_student.nunique()
n_multi = int((student_info.groupby("id_student").size() > 1).sum())
print(f"student-presentation rows : {n_rows:,}")
print(f"distinct students         : {n_people:,}")
print(f"students in >1 presentation: {n_multi:,} ({n_multi / n_people:.1%} of people)")
print(f"modules / presentations   : {courses.code_module.nunique()} / {len(courses)}")
print(f"VLE click log rows        : {n_click_rows:,}")

student-presentation rows : 32,593
distinct students         : 28,785
students in >1 presentation: 3,538 (12.3% of people)
modules / presentations   : 7 / 22
VLE click log rows        : 10,655,280


**Documentation inconsistency found.** The official `OULAD.names` file lists *32,953* students in section 5 but its class table
totals *32,593*. The data contain 32,593 rows, so the 32,953 figure is a typo in the metadata. The transformation log in notebook 02 records this.

## 4. Course-presentations

In [5]:
timeline = (student_info.groupby(["code_module", "code_presentation"]).size().unstack(fill_value=0))
timeline["total"] = timeline.sum(axis=1)
timeline

code_presentation,2013B,2013J,2014B,2014J,total
code_module,,,,,
AAA,0,383,0,365,748
BBB,1767,2237,1613,2292,7909
CCC,0,0,1936,2498,4434
DDD,1303,1938,1228,1803,6272
EEE,0,1052,694,1188,2934
FFF,1614,2283,1500,2365,7762
GGG,0,952,833,749,2534


Not every module runs in every presentation (for example module AAA has no February "B" presentations, module CCC has no 2013 presentations, and modules EEE and GGG have no 2013B).
For the temporal split planned later (train on 2013, test on 2014), this means module mix differs between train and test.
That is a first source of **data drift** and must be handled explicitly, for example by reporting results per module.

## 5. Next step

Notebook 02 builds the analysis table: one row per eligible student-presentation, with features restricted to the first 28 days.